# Black–Scholes–Merton 유럽형 옵션 가격 모델 구현 설명

이 문서는 Black–Scholes–Merton(BSM) PDE의 closed-form solution을 이용해 유럽형 call/put 가격을 계산하도록 추가한 코드의 **전체 구조, 금융 가정, 구현 흐름, 경계조건, 입력 검증, 단위 테스트, 연구 노트북 구성**을 설명한다.

이번 작업에서 생성한 파일은 다음과 같다.

- 재사용 가격 함수: [bsm.py](../src/option_pricing_volatility/models/bsm.py)
- 최소 단위 테스트: [test_bsm.py](../tests/test_bsm.py)
- BSM benchmark 연구 노트북: [04_01_BSM_model.ipynb](../notebooks/04_bsm/04_01_BSM_model.ipynb)

구현이 따르는 기존 계약은 [model_contracts.md](model_contracts.md)의 BSM 절에 정의되어 있다. 계약 문서에 공식과 경계조건이 이미 명시되어 있었으므로 이번 작업은 계약을 변경하지 않고 그대로 코드로 옮겼다. 새로운 dependency, package export, helper class는 추가하지 않았다.

## 1. 전체 구조와 데이터 흐름

<pre>
연구 노트북 또는 다른 패키지 코드
        |
        | spot, strike, maturity, rate,
        | volatility, option_type, dividend_yield
        v
src/option_pricing_volatility/models/bsm.py
        |
        |-- 모든 입력 검증
        |-- Python float으로 정규화
        |-- T = 0 경계 처리
        |-- sigma = 0 경계 처리
        |-- d1, d2 계산
        |-- call/put closed-form 공식 평가
        v
단일 float 옵션 가격
        |
        +--> tests/test_bsm.py에서 금융 계약 검증
        |
        +--> notebooks/04_bsm/04_01_BSM_model.ipynb에서
             option_type | bsm_price tidy table 생성
</pre>

각 파일의 책임은 분리되어 있다.

| 파일 | 책임 | 포함하지 않는 것 |
|---|---|---|
| <code>models/bsm.py</code> | 하나의 유럽형 call 또는 put 가격 계산 | DataFrame, plotting, 반복 실험 |
| <code>tests/test_bsm.py</code> | 공식, parity, 경계조건, 입력 계약 보호 | 대규모 parameter grid, 성능 시험 |
| <code>04_01_BSM_model.ipynb</code> | 가정과 공식 설명, synthetic 입력 설정, tidy 결과 구성 | 가격 공식의 재구현, 최종 수렴 그래프 |

이 분리를 통해 CRR Binomial Tree와 GBM Monte Carlo가 향후 동일한 입력에서 <code>bsm_price()</code>를 분석적 benchmark로 재사용할 수 있다.

## 2. 금융 모형과 closed-form solution

가격 함수는 BSM PDE를 수치적으로 푸는 것이 아니라 그 PDE의 closed-form solution을 직접 평가한다. 달력시간을 $t$, 계약 만기시각을 $T_{\mathrm{exp}}$라 하면 연속배당수익률 $q$를 포함한 BSM PDE는

$$
\frac{\partial V}{\partial t}
+\frac12\sigma^2S^2\frac{\partial^2V}{\partial S^2}
+(r-q)S\frac{\partial V}{\partial S}
-rV=0
$$

이며 만기조건은

$$
C(S,T_{\mathrm{exp}})=\max(S-K,0),
\qquad
P(S,T_{\mathrm{exp}})=\max(K-S,0)
$$

이다. 잔존만기는 $\tau=T_{\mathrm{exp}}-t$이며, 이 저장소의 API와 아래 closed-form 공식에서는 관례에 따라 잔존만기 $\tau$를 $T$로 표기한다. 구현은 PDE 시간·가격 격자를 만들지 않고 이 만기조건에 대응하는 분석해를 사용한다. 핵심 가정은 다음과 같다.

- 옵션은 European call 또는 put이며 만기에만 행사하고, 가격 계약상 만기에 현금결제한다.
- 기초자산은 위험중립측도에서 연속배당수익률을 포함한 GBM을 따른다.
- 무위험이자율 $r$, 배당수익률 $q$, 변동성 $\sigma$는 만기까지 일정하다.
- $r$과 $q$는 연속복리 연율의 소수 단위다.
- $\sigma$는 연율 변동성의 소수 단위이고, $T$는 연 단위다. 날짜에서 $T$를 만들 때의 기본 year fraction은 ACT/365F이며 함수 자체는 날짜 변환을 수행하지 않는다.
- 현물가격 $S$와 행사가격 $K$는 같은 통화 단위를 사용한다.
- 거래비용과 세금 등 시장 마찰은 고려하지 않는다.

$T>0$이고 $\sigma>0$일 때

$$
d_1 = \frac{\ln(S/K)+(r-q+\tfrac12\sigma^2)T}{\sigma\sqrt{T}},
\qquad
d_2 = d_1-\sigma\sqrt{T}
$$

이며, $N(\cdot)$을 표준정규 누적분포함수라고 하면

$$
C = Se^{-qT}N(d_1)-Ke^{-rT}N(d_2)
$$

$$
P = Ke^{-rT}N(-d_2)-Se^{-qT}N(-d_1)
$$

이다. 배당이 있는 경우의 put–call parity는

$$
C-P=Se^{-qT}-Ke^{-rT}
$$

이며 단위 테스트가 이 관계를 직접 확인한다.

유럽형 옵션의 no-arbitrage bounds는

$$
\max(0,Se^{-qT}-Ke^{-rT})\le C\le Se^{-qT}
$$

$$
\max(0,Ke^{-rT}-Se^{-qT})\le P\le Ke^{-rT}
$$

이다. closed-form 가격은 유효한 입력에서 이 범위를 만족한다. 현재 최소 구현은 계산된 가격을 이 범위로 clipping하거나 별도의 사후 bound 검사로 보정하지 않으며, 최소 테스트 범위에서도 bounds를 별도 parameter grid로 검증하지 않는다.

## 3. 공개 API와 입력 단위

공개 함수의 signature는 다음과 같다. 아래 표시는 Markdown 안의 설명용 코드이며 이 문서 노트북에는 실행 코드 셀이 없다.

<pre><code class="language-python">def bsm_price(
    spot: float,
    strike: float,
    maturity: float,
    rate: float,
    volatility: float,
    option_type: str,
    dividend_yield: float = 0.0,
) -&gt; float:</code></pre>

| 인수 | 금융 의미 | 허용 규칙 |
|---|---|---|
| <code>spot</code> | 현재 기초자산 가격 $S$ | 유한한 실수, $S>0$ |
| <code>strike</code> | 행사가격 $K$ | 유한한 실수, $K>0$ |
| <code>maturity</code> | 잔존만기 $T$ | 유한한 실수, 연 단위, $T\ge0$ |
| <code>rate</code> | 연속복리 무위험이자율 $r$ | 유한한 실수, 소수 단위 |
| <code>volatility</code> | 연율 변동성 $\sigma$ | 유한한 실수, 소수 단위, $\sigma\ge0$ |
| <code>option_type</code> | payoff 종류 | 정확히 <code>call</code> 또는 <code>put</code> |
| <code>dividend_yield</code> | 연속복리 배당수익률 $q$ | 유한한 실수, 소수 단위, 기본값 0 |

유한한 음의 무위험이자율과 배당수익률은 금융적으로 가능할 수 있으므로 별도의 부호 제한을 두지 않는다. 함수는 배열이나 Series가 아니라 한 세트의 scalar 입력만 받고 항상 Python <code>float</code> 가격 하나를 반환한다. 대소문자 변환이나 별칭은 수행하지 않으므로 <code>Call</code>, <code>CALL</code>, <code>digital</code> 등은 허용하지 않는다.

## 4. 모듈 구성과 입력 검증 코드

모듈은 표준 라이브러리만 사용한다.

<pre><code class="language-python">import math
from numbers import Real
from statistics import NormalDist</code></pre>

- <code>math</code>: 유한성 검사, 지수함수, 로그, 제곱근을 계산한다.
- <code>Real</code>: Python <code>float</code>/<code>int</code>뿐 아니라 실수형 scalar라는 의미를 검사한다.
- <code>NormalDist</code>: SciPy 없이 표준정규 CDF를 계산한다.

금융 수치 입력은 dictionary로 모은 뒤 동일한 규칙으로 검사한다.

<pre><code class="language-python">numeric_inputs = {
    "spot": spot,
    "strike": strike,
    "maturity": maturity,
    "rate": rate,
    "volatility": volatility,
    "dividend_yield": dividend_yield,
}
for name, value in numeric_inputs.items():
    if (
        isinstance(value, bool)
        or not isinstance(value, Real)
        or not math.isfinite(value)
    ):
        raise ValueError(f"{name} must be a finite real number")</code></pre>

Python에서 <code>bool</code>은 <code>int</code>의 하위 타입이므로 <code>isinstance(True, Real)</code>이 참이 될 수 있다. 가격이나 만기에 <code>True</code>가 숫자 1처럼 조용히 사용되지 않도록 <code>bool</code>을 먼저 명시적으로 거부한다. <code>NaN</code>과 양·음의 무한대도 <code>math.isfinite()</code>로 거부한다.

유한성 검사 후 금융 도메인과 옵션 종류를 검사한다.

<pre><code class="language-python">if spot &lt;= 0:
    raise ValueError("spot must be greater than 0")
if strike &lt;= 0:
    raise ValueError("strike must be greater than 0")
if maturity &lt; 0:
    raise ValueError("maturity must be greater than or equal to 0")
if volatility &lt; 0:
    raise ValueError("volatility must be greater than or equal to 0")
if not isinstance(option_type, str) or option_type not in {"call", "put"}:
    raise ValueError("option_type must be 'call' or 'put'")</code></pre>

검증을 통과한 수치 입력은 모두 내장 <code>float</code>으로 변환한다. 이는 정수나 호환되는 실수 scalar가 들어와도 이후 연산과 반환형을 일관되게 유지하기 위한 단계다. 모든 검증은 경계조건 반환보다 먼저 실행되므로 $T=0$이나 $\sigma=0$이어도 다른 잘못된 입력이 숨겨지지 않는다.

## 5. 경계조건 처리

일반 BSM 식의 분모에는 $\sigma\sqrt{T}$가 있으므로 $T=0$ 또는 $\sigma=0$을 그대로 대입하면 0으로 나누게 된다. 구현은 작은 양의 값으로 강제 보정하지 않고 경제적 의미가 분명한 별도 공식을 사용한다.

### 5.1 만기 $T=0$

현재가 바로 만기이므로 할인이나 확률분포 없이 내재가치를 반환한다.

$$
C=\max(S-K,0),
\qquad
P=\max(K-S,0)
$$

<pre><code class="language-python">if maturity == 0:
    payoff = spot - strike if option_type == "call" else strike - spot
    return float(max(payoff, 0.0))</code></pre>

### 5.2 무변동성 $\sigma=0$, $T>0$

위험중립 기초자산 경로가 결정론적이므로 할인된 현물과 행사가격의 차이를 직접 계산한다.

$$
C=\max(Se^{-qT}-Ke^{-rT},0)
$$

$$
P=\max(Ke^{-rT}-Se^{-qT},0)
$$

<pre><code class="language-python">discounted_spot = spot * math.exp(-dividend_yield * maturity)
discounted_strike = strike * math.exp(-rate * maturity)
if volatility == 0:
    payoff = (
        discounted_spot - discounted_strike
        if option_type == "call"
        else discounted_strike - discounted_spot
    )
    return float(max(payoff, 0.0))</code></pre>

분기 순서는 $T=0$이 먼저이고 $\sigma=0$이 다음이다. 두 값이 동시에 0이면 만기 내재가치가 우선하며, 이는 CRR 및 Monte Carlo 함수의 기존 경계 규약과 같다.

## 6. 일반적인 BSM 계산 코드

$T>0$, $\sigma>0$이면 먼저 반복되는 $\sigma\sqrt{T}$를 <code>volatility_time</code>에 저장하고 계약의 $d_1$, $d_2$를 그대로 계산한다.

<pre><code class="language-python">volatility_time = volatility * math.sqrt(maturity)
d1 = (
    math.log(spot / strike)
    + (rate - dividend_yield + 0.5 * volatility**2) * maturity
) / volatility_time
d2 = d1 - volatility_time
normal = NormalDist()</code></pre>

코드와 수식의 대응은 다음과 같다.

| 코드 표현 | 수학 표현 | 의미 |
|---|---|---|
| <code>math.log(spot / strike)</code> | $\ln(S/K)$ | log-moneyness |
| <code>rate - dividend_yield</code> | $r-q$ | 위험중립 carry |
| <code>0.5 * volatility**2</code> | $\tfrac12\sigma^2$ | 로그정규 보정항 |
| <code>volatility_time</code> | $\sigma\sqrt{T}$ | 만기까지의 총 표준편차 |
| <code>normal.cdf(x)</code> | $N(x)$ | 표준정규 누적확률 |

call과 put은 각각의 closed-form 식을 직접 평가한다.

<pre><code class="language-python">if option_type == "call":
    price = (
        discounted_spot * normal.cdf(d1)
        - discounted_strike * normal.cdf(d2)
    )
else:
    price = (
        discounted_strike * normal.cdf(-d2)
        - discounted_spot * normal.cdf(-d1)
    )

return float(price)</code></pre>

<code>discounted_spot</code>은 $Se^{-qT}$, <code>discounted_strike</code>는 $Ke^{-rT}$에 정확히 대응한다. 이미 검증 단계에서 option type이 두 값 중 하나임을 보장했으므로 마지막 <code>else</code>는 put 분기다. 계산 결과를 다시 <code>float</code>으로 감싸 공개 반환형을 명시적으로 고정한다.

## 7. 수치 및 설계 선택

### 표준정규 CDF

표준정규 CDF는 <code>statistics.NormalDist().cdf</code>를 사용한다. SciPy 같은 새 dependency를 도입하지 않으면서 Python 표준 라이브러리만으로 필요한 closed-form 계산을 완성하기 위한 선택이다.

### scalar API

함수는 의도적으로 scalar 전용이다. NumPy 배열의 우연한 broadcasting이나 암묵적 vectorization에 의존하지 않는다. 여러 옵션을 계산할 때는 호출자가 반복하거나 별도의 후속 API를 설계해야 한다.

### 계산 복잡도

트리나 난수표본을 만들지 않고 고정된 개수의 산술연산과 CDF 평가만 수행하므로 한 옵션의 시간·추가 메모리 복잡도는 모두 $O(1)$이다. 따라서 CRR discretization error나 Monte Carlo sampling error가 없는 분석적 benchmark 역할을 한다.

### 예외와 clipping

잘못된 입력은 <code>ValueError</code>로 명시적으로 거부하며 가격, 확률, 변동성을 유효 범위로 조용히 clipping하지 않는다. $T=0$과 $\sigma=0$만 계약에 정의된 경제적 경계 공식으로 처리한다.

### 현재 수치 범위

최소 구현 범위에 맞추어 극단적으로 큰 유한 $r$, $q$, $T$에서 <code>exp</code> overflow가 발생하는 경우나 극단적 $S/K$ 비율을 위한 별도 안정화 알고리즘은 추가하지 않았다. 이는 기존 CRR/Monte Carlo 구현과 같은 수준의 입력 규약이며, 필요할 경우 별도의 수치 안정성 작업으로 다뤄야 한다.

## 8. 단위 테스트의 구조와 의도

<code>tests/test_bsm.py</code>는 요청된 최소 금융 계약만 검증하며 대규모 parameter grid는 만들지 않는다. parameterization을 전개하면 benchmark 2개, parity 1개, $T=0$ 2개, $\sigma=0$ 2개, invalid input 7개로 총 14개 테스트가 수집된다.

| 테스트 범주 | 입력 또는 관계 | 보호하는 동작 |
|---|---|---|
| 표준 benchmark | $S=K=100$, $T=1$, $r=0.05$, $q=0$, $\sigma=0.2$ | 알려진 call/put 가격 재현 |
| 배당 포함 parity | $C-P=Se^{-qT}-Ke^{-rT}$ | $q$와 할인항의 부호 및 call/put 공식 검증 |
| $T=0$ | call과 put | 내재가치 경계 검증 |
| $\sigma=0$ | call과 put | 할인된 결정론적 payoff 검증 |
| 잘못된 입력 | 도메인, 비유한값, option type | <code>ValueError</code> 계약 검증 |

### 8.1 알려진 표준 가격

표준 입력에서 독립적으로 알려진 기준값은 다음과 같다.

- call: <code>10.450583572185565</code>
- put: <code>5.573526022256971</code>

이 입력에서는 $d_1=0.35$, $d_2=0.15$다.

두 옵션 종류를 <code>pytest.mark.parametrize</code>로 반복하고 <code>pytest.approx</code>로 부동소수점 오차를 허용해 비교한다. call 기준값은 기존 CRR 및 Monte Carlo 테스트에서, put 기준값은 기존 Monte Carlo 테스트에서 각각 수렴·통계 일관성의 기준으로 사용되던 값이다.

### 8.2 배당 포함 put–call parity

테스트는 $S=100$, $K=105$, $T=1.25$, $r=0.04$, $q=0.015$, $\sigma=0.25$에서 call과 put을 각각 계산한 뒤

$$
C-P \approx Se^{-qT}-Ke^{-rT}
$$

를 확인한다. 이 한 관계는 두 가격식에 동일한 할인계수와 배당수익률이 일관되게 적용되었는지 검증한다.

### 8.3 경계조건

$T=0$ 테스트는 $S=105$, $K=100$에서 call 5, put 0을 확인한다. $\sigma=0$ 테스트는 코드와 별도로 할인된 forward value를 계산하고 call은 그 양의 부분, put은 음의 부분의 양수를 기대값으로 사용한다.

### 8.4 잘못된 입력

테스트 입력 dictionary에서 필드 하나씩을 바꾸어 <code>spot=0</code>, 음의 strike, 음의 maturity, 음의 volatility, 무한 rate, NaN dividend yield, <code>digital</code> option type을 거부하는지 확인한다. 모든 경우 공개 함수가 <code>ValueError</code>를 발생시켜야 한다.

## 9. BSM 연구 노트북의 구성

<code>notebooks/04_bsm/04_01_BSM_model.ipynb</code>는 재사용 코드를 담는 장소가 아니라 설명, 파라미터 설정, 결과 정리를 담당한다. 셀 구성은 다음과 같다.

1. BSM PDE의 closed-form solution을 사용한다는 설명과 핵심 가정
2. $d_1$, $d_2$, call/put 공식의 Markdown 설명
3. pandas와 패키지의 <code>bsm_price</code> import
4. 기존 CRR/GBM-MC 노트북과 동일한 synthetic 입력으로 tidy 결과 생성

공통 synthetic 입력은 다음과 같다.

<pre><code class="language-python">synthetic_parameters = {
    "spot": 100.0,
    "strike": 100.0,
    "maturity": 1.0,
    "rate": 0.05,
    "volatility": 0.2,
    "dividend_yield": 0.0,
}</code></pre>

노트북은 BSM 공식을 별도 함수로 다시 구현하지 않고 다음 package import를 사용한다.

<pre><code class="language-python">from option_pricing_volatility.models.bsm import bsm_price</code></pre>

call과 put을 반복해 명시적인 열 순서로 DataFrame을 만든다.

<pre><code class="language-python">bsm_results = pd.DataFrame(
    [
        {
            "option_type": option_type,
            "bsm_price": bsm_price(
                **synthetic_parameters,
                option_type=option_type,
            ),
        }
        for option_type in ("call", "put")
    ],
    columns=["option_type", "bsm_price"],
)</code></pre>

재실행되어 저장된 결과는 다음과 같다.

| option_type | bsm_price |
|---|---:|
| call | 10.450584 |
| put | 5.573526 |

<code>option_type</code>이 merge key이고 가격 열 이름이 <code>bsm_price</code>이므로 기존 CRR 및 GBM-MC 수렴 결과에 benchmark를 바로 연결할 수 있다. 이번 단계에서는 실제 merge, 오차 열, 그래프를 만들지 않았다.

## 10. 실행한 검증과 결과

구현 후 다음 검증을 수행했다.

<pre><code class="language-text">python -m pytest tests/test_bsm.py
# 14 passed

python -m pytest
# 74 passed</code></pre>

좁은 테스트는 새 BSM 계약만 빠르게 확인했고, 전체 테스트는 기존 market data, CRR, GBM, Monte Carlo 동작에 회귀가 없는지 확인했다. 또한 <code>git diff --check</code>가 통과했다.

BSM 연구 노트북은 기존 <code>finance</code> 커널로 위에서 아래까지 재실행했다. 두 코드 셀의 execution count가 순서대로 기록되었고 마지막 셀에 call/put 두 행이 오류 없이 저장되었다. 새 dependency는 설치하지 않았다.

저장소에는 별도의 lint, formatter, type checker, notebook CI 설정이 없어 추가 검사 명령은 실행하지 않았다.

## 11. 현재 범위, 한계, 후속 연결

현재 구현에 포함된 범위는 다음과 같다.

- scalar European call/put BSM 가격
- 연속배당수익률 $q$ 포함
- $T=0$과 $\sigma=0$의 명시적 경제적 경계
- 기존 CRR/Monte Carlo 방식과 일관된 입력 검증
- SciPy 없는 표준정규 CDF
- 최소 단위 테스트와 2행 tidy benchmark 결과

의도적으로 포함하지 않은 범위는 다음과 같다.

- CRR 또는 GBM-Monte Carlo와의 최종 수렴 비교 및 그래프
- delta, gamma, vega 등 Greeks
- implied volatility solver
- BSM PDE의 finite-difference 등 수치해법
- American, Asian, barrier 등 다른 exercise style 또는 exotic payoff
- 배열 입력을 위한 vectorized API
- 극단적 입력 전용 overflow·underflow 안정화

향후 비교 단계에서는 같은 <code>option_type</code>에 대해 BSM 가격을 CRR/MC tidy 결과와 병합한다. CRR과 BSM의 차이는 주로 시간 격자 discretization error로, MC와 BSM의 차이는 sampling error와 함께 해석해야 한다. BSM 값은 closed-form analytical benchmark이므로 두 수치 방법이 수렴해야 할 공통 기준점 역할을 한다.